In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS azuredatabricks_catalog.silver

# Bronze Customers Table

In [0]:
df_customers = spark.read.table("azuredatabricks_catalog.bronze.customers")

display(df_customers)

In [0]:
# Dropping Nulls
# display(df_customers.dropna(subset=['country']))

# Replacing Nulls
df_customers = df_customers.fillna("Unknown")
df_customers = df_customers.fillna(0,subset=['age'])
display(df_customers)



In [0]:
df_customers = df_customers.dropDuplicates()

In [0]:
# df_india_usa = df_customers.filter((col("country") == "India") |  (col("country") == "USA") )
# display(df_india_usa)

In [0]:
# df_india_usa_not = df_customers.filter((col("country") != "India") &  (col("country") != "USA") )
# display(df_india_usa_not)

In [0]:
# df_combined = df_india_usa.unionByName(df_india_usa_not)
# display(df_combined)

In [0]:
# display(df_india_usa.intersect(df_customers).count())

In [0]:
# df_except = df_customers.exceptAll(df_india_usa)
# display(df_except)

In [0]:
display(df_customers)

In [0]:
df_customers = df_customers.select(col("customer_id").alias("customer_id"), "customer_name", "email", "country", "signup_date", "customer_type", "age", "updated_at")


In [0]:
df_customers = df_customers.withColumn("silver_processed_at", current_timestamp())

In [0]:
df_customers = df_customers.withColumn("domains",split(col("email"),"@")[1])


In [0]:
df_customers = df_customers.withColumn("signup_date",col("signup_date").cast(TimestampType()))


In [0]:
df_customers = df_customers.withColumn("flag",when(col("age")<15,lit("Junior"))\
                                            .when((col("age")>=15) & (col("age")<35),lit("Middle"))\
                                            .otherwise(lit("Senior")))


In [0]:
# Data Writing [UPSERT]

from delta.tables import DeltaTable 

# Checking if table exists
if spark.catalog.tableExists("azuredatabricks_catalog.silver.customers"):
    delta_obj = DeltaTable.forName(spark,"azuredatabricks_catalog.silver.customers")


    delta_obj.alias("trg").merge(df_customers.alias("src"), 
                                "trg.customer_id = src.customer_id")\
                            .whenMatchedUpdateAll(condition="src.updated_at > trg.updated_at")\
                            .whenNotMatchedInsertAll()\
                            .execute()

else:
    df_customers.write.format("delta")\
            .mode("overwrite")\
            .saveAsTable("azuredatabricks_catalog.silver.customers")


# Bronze Products

In [0]:
df_prod = spark.read.table("azuredatabricks_catalog.bronze.products")

df_prod = df_prod.fillna("Unknown")
df_prod = df_prod.fillna(0,subset=["price"])


df_prod = df_prod.dropDuplicates()

df_prod = df_prod.withColumn("silver_processed_at", current_timestamp())

display(df_prod)


In [0]:
# Data Writing [UPSERT]

from delta.tables import DeltaTable 

# Checking if table exists
if spark.catalog.tableExists("azuredatabricks_catalog.silver.products"):
    delta_obj = DeltaTable.forName(spark,"azuredatabricks_catalog.silver.products")

    delta_obj.alias("trg").merge(df_prod.alias("src"), 
                                "trg.product_id = src.product_id")\
                            .whenMatchedUpdateAll(condition="src.updated_at > trg.updated_at")\
                            .whenNotMatchedInsertAll()\
                            .execute()

else:
    df_prod.write.format("delta")\
            .mode("overwrite")\
            .saveAsTable("azuredatabricks_catalog.silver.products")


In [0]:
%sql
select count(*) from azuredatabricks_catalog.silver.products

# Bronze Orders Table

In [0]:
df_orders = spark.read.table("azuredatabricks_catalog.bronze.orders")

df_orders = df_orders.fillna(0,subset=["discount","order_amount","quantity"])
df_orders = df_orders.fillna("Unknown",subset=["payment_status"])
df_orders = df_orders.fillna("9999-01-01",subset=["order_date"])

df_orders = df_orders.dropDuplicates()

df_orders = df_orders.withColumn("silver_processed_at", current_timestamp())

display(df_orders)

In [0]:
df_orders = df_orders.withColumn("order_date",coalesce(try_to_date(col("order_date"),"yyyy-MM-dd"),try_to_date(col("order_date"),"dd-MM-yyyy")))

df_orders = df_orders.withColumn("order_date",col("order_date").cast(TimestampType()))

df_orders = df_orders.withColumn("updated_at",to_timestamp(from_unixtime(col("updated_at")/1000)))

display(df_orders)

In [0]:
# Data Writing [UPSERT]

from delta.tables import DeltaTable 

# Checking if table exists
if spark.catalog.tableExists("azuredatabricks_catalog.silver.orders"):
    delta_obj = DeltaTable.forName(spark,"azuredatabricks_catalog.silver.orders")

    delta_obj.alias("trg").merge(df_orders.alias("src"), 
                                "trg.order_id = src.order_id")\
                            .whenMatchedUpdateAll(condition="src.updated_at > trg.updated_at")\
                            .whenNotMatchedInsertAll()\
                            .execute()

else:
    df_orders.write.format("delta")\
            .mode("overwrite")\
            .saveAsTable("azuredatabricks_catalog.silver.orders")


In [0]:
%sql
SELECT count(*) FROM azuredatabricks_catalog.silver.orders